# Phase 1: ERM baseline on Waterbirds (ResNet-18)

**Kaggle settings:** Accelerator = GPU (T4 or P100). Turn **Internet on** if you want to download Waterbirds instead of attaching it as an input dataset.

This notebook:
1. sets up the project code,
2. finds or downloads Waterbirds and checks it against the official split and group counts,
3. trains the ERM baseline for 3 seeds (checkpoint chosen by **validation worst-group accuracy**),
4. summarizes test results (mean ± std over seeds).

## 1. Project code
Either upload the `ann` folder as a Kaggle dataset (it is read-only, so it gets copied to `/kaggle/working`) or set `GIT_URL` to your repo.

In [ ]:
import os, shutil, subprocess, glob

GIT_URL = "https://github.com/aarush-luthra/compression-group-robustness.git"  # set "" to use an uploaded dataset instead
WORK = "/kaggle/working/ann" if os.path.isdir("/kaggle") else os.path.abspath("..")

if os.path.isdir("/kaggle") and not os.path.isdir(WORK):
    if GIT_URL:
        subprocess.run(["git", "clone", GIT_URL, WORK], check=True)
    else:
        src = [os.path.dirname(os.path.dirname(p)) for p in glob.glob("/kaggle/input/**/src/train_erm.py", recursive=True)]
        assert src, "Upload the project as a Kaggle dataset or set GIT_URL"
        shutil.copytree(src[0], WORK)
os.chdir(WORK)
print("Working dir:", os.getcwd())

## 2. Dataset: find or download, then verify

In [ ]:
# If Waterbirds is attached as an input dataset, use it; otherwise download from the Hugging Face mirror grodino/waterbirds (~492 MB, needs Internet on).
found = glob.glob("/kaggle/input/**/metadata.csv", recursive=True)
found = [os.path.dirname(p) for p in found if "waterbird" in p.lower() or os.path.isdir(os.path.join(os.path.dirname(p), "001.Black_footed_Albatross"))]
if found:
    DATA_ROOT, DOWNLOAD = found[0], False
else:
    DATA_ROOT, DOWNLOAD = ("/kaggle/working/data" if os.path.isdir("/kaggle") else "data/waterbirds"), True
print("DATA_ROOT:", DATA_ROOT, "| download:", DOWNLOAD)

In [ ]:
from src.prepare_data import download_hf, find_root, verify

# Hugging Face mirror (official Stanford host is often unreachable)
root = download_hf(DATA_ROOT) if DOWNLOAD else DATA_ROOT
DATA_ROOT = find_root(root)
errors = verify(DATA_ROOT)
assert not errors, errors
print("Verified dataset at", DATA_ROOT)

## 3. Train ERM baseline (3 seeds)
Each run selects the epoch with the best **val** worst-group accuracy and evaluates test exactly once. Rows are appended to `results/results.csv`.

In [ ]:
SEEDS = [0, 1, 2]
EPOCHS = 30

for s in SEEDS:
    print(f"===== seed {s} =====")
    subprocess.run(["python", "-m", "src.train_erm", "--data_root", DATA_ROOT,
                    "--seed", str(s), "--epochs", str(EPOCHS)], check=True)

## 4. Results

In [ ]:
import pandas as pd

df = pd.read_csv("results/results.csv")
test = df[(df.model == "erm") & (df.split == "test")]
cols = ["overall", "g0", "g1", "g2", "g3", "wga", "size_mb", "latency_ms"]
display(test[["seed"] + cols])
summary = test[cols].agg(["mean", "std"]).T
summary.columns = ["mean", "std"]
display(summary.round(4))

In [ ]:
import matplotlib.pyplot as plt

names = ["landbird/land", "landbird/water*", "waterbird/land*", "waterbird/water"]
m = test[["g0", "g1", "g2", "g3"]].mean() * 100
sd = test[["g0", "g1", "g2", "g3"]].std() * 100
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(names, m, yerr=sd, capsize=4)
ax.axhline(test["overall"].mean() * 100, ls="--", c="k", label="overall accuracy")
ax.set_ylabel("Test accuracy (%)"); ax.set_ylim(0, 100)
ax.set_title("ERM ResNet-18: per-group test accuracy (* = rare group)")
ax.legend(); plt.xticks(rotation=15); plt.tight_layout()
os.makedirs("results", exist_ok=True)
plt.savefig("results/erm_group_acc.png", dpi=150); plt.show()

**Sanity check:** expect about 97% overall accuracy and roughly 60–75% worst-group accuracy (usually waterbird/land). A worst-group accuracy near 90% for plain ERM points to a leak or split bug.

Save `checkpoints/erm_seed*.pt`. They are the teachers and the starting point for pruning and quantization in Phase 2.